# Case 4 · Should we badge more sellers? (Part 2)
**BUS-G 492 · Causal decisions with AI**

In [ ]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case4_panel_badge/data/"
for candidate in ["data/", "cases/case4_panel_badge/data/", "../cases/case4_panel_badge/data/", "../../../cases/case4_panel_badge/data/", "../../../../cases/case4_panel_badge/data/"]:
    if os.path.exists(candidate + "seller_months.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

## Stage 5 · Something you didn't know

> **From:** Sara Lindqvist, Trust & Safety
> **Re:** how badges are actually granted
>
> We don't hand badges out at random. Our reviewers badge sellers they consider **excellent**: consistent quality, fast shipping, great service. Those sellers were usually doing very well **before** they got the badge.
>
> The **month** a seller gets the badge mostly depends on when a reviewer reaches their application in our **queue**, which can take anywhere from a few weeks to a year and has nothing to do with the seller's sales that month. Grant dates: `badge_grants.csv`.

### Load the data

This is a new notebook, so nothing from Part 1 is loaded yet. Run this cell to load the Part 1 data again, then the next cell for the new files.

In [ ]:
%pip install -q linearmodels   # not preinstalled in Colab
import statsmodels.formula.api as smf
import pandas as pd, numpy as np
df = pd.read_csv(DATA + "seller_months.csv")
print(df.shape)
df.head()

In [ ]:
grants = pd.read_csv(DATA + "badge_grants.csv")
grants.head()

## Your tasks

Use AI again, but this time **tell it the whole story**: how the data were generated, and what the new information is. Each step ends with a table: fill it in (double-click the cell to type) before moving on.

### Step 1 · Were badged sellers already big before the badge?

The code below lines up every seller who got the badge **during** 2023–2025 by months relative to their badge month (`rel`: −1 = the month before, 0 = the badge month). Fill in:

| Average monthly revenue | \$ |
|---|---|
| Badged sellers, 12 months **before** the badge (rel −12 to −1) | |
| Badged sellers, 12 months **after** the badge (rel 0 to 11) | |
| Sellers who were **never** badged (all months) | |

What does the first row tell you about the Part 1 comparison?

In [ ]:
import numpy as np
g = grants[grants.badge_granted.notna() & (grants.badge_granted != "before 2023")].copy()
g["gm"] = pd.PeriodIndex(g.badge_granted, freq="M")
e = df.merge(g[["seller_id", "gm"]], on="seller_id")
e["rel"] = pd.PeriodIndex(e.month, freq="M").asi8 - pd.PeriodIndex(e.gm, freq="M").asi8
never = df[~df.seller_id.isin(grants[grants.badge_granted.notna()].seller_id)]
# Your code here


### Step 2 · Look back: compare each seller with itself (fixed effects) → ETT

Steps 1–3 use only the **2023–2025 data** to estimate what the badge did for the sellers who got it: the effect on the treated (ETT). Nothing about 2026 yet.

Estimate the badge effect on **log revenue** three ways and report `exp(coefficient) − 1`:

| Method | Badge effect on revenue |
|---|---|
| Pooled regression (no controls) | |
| + rating, listings, category, price tier, month | |
| **Seller + month fixed effects (ETT)** | |

In plain words: what do seller fixed effects remove? What do month fixed effects remove?

In [ ]:
from linearmodels.panel import PanelOLS
df["lr"] = np.log(df.revenue)
# pooled and controls: smf.ols("lr ~ has_badge", df), then add avg_rating + np.log(active_listings) + C(category) + C(price_tier) + C(month)
# Your code here

p = df.set_index(["seller_id", "month"])
p.index = p.index.set_levels([p.index.levels[0], pd.to_datetime(p.index.levels[1])])
fe = PanelOLS.from_formula("lr ~ has_badge + EntityEffects + TimeEffects", p).fit(cov_type="clustered", cluster_entity=True)
print(f"seller + month FE: {np.exp(fe.params['has_badge']) - 1:+.1%}")


### Step 3 · Look back: check for a trend before the badge (event study)

The code below estimates the badge effect separately for each month from 12 before to 12 after the badge (−1 is the reference). Plot the coefficients, then fill in:

| Months relative to the badge | Average coefficient |
|---|---|
| −12 to −2 (before) | |
| 0 to +12 (after) | |

Is there a trend before the badge? What would a rising trend before the badge mean for your Step 2 estimate?

In [ ]:
e2 = df.merge(g[["seller_id", "gm"]], on="seller_id", how="left")
rel = pd.PeriodIndex(e2.month, freq="M").asi8 - pd.PeriodIndex(e2.gm, freq="M").asi8
e2["rel"] = np.where(e2.gm.isna(), np.nan, np.clip(rel, -12, 12))
cols = []
for k in range(-12, 13):
    if k == -1:
        continue
    c = f"r{k + 12:02d}"; e2[c] = (e2.rel == k).astype(int); cols.append((k, c))
early = grants[grants.badge_granted == "before 2023"].seller_id
ep = e2[~e2.seller_id.isin(early)].set_index(["seller_id", "month"])
ep.index = ep.index.set_levels([ep.index.levels[0], pd.to_datetime(ep.index.levels[1])])
es = PanelOLS(ep.lr, ep[[c for _, c in cols]], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
coefs = pd.Series({k: es.params[c] for k, c in cols})
print(coefs.round(3))
# Your code here: plot coefs, and average them before and after


### Step 4 · Look ahead: predict 2026 profit → choose R

Now **predict** what badging more sellers would earn in 2026: apply the effect you estimated in Step 2 to sellers who don't have the badge yet, using their 2025 revenue. This is a forecast: your deployment will show the actual 2026 result.

CraftHub earns **12%** of extra sales and pays **\$300 a year** per badge. With your ETT from Step 2:

| Seller's monthly revenue | CraftHub's gain per year = 0.12 × ETT × 12 × revenue | Gain − \$300 | Badge? |
|---|---|---|---|
| \$1,000 | | | |
| \$2,000 | | | |
| \$3,000 | | | |
| \$4,000 | | | |
| \$6,000 | | | |

Choose R. Predicted profit = Σ (gain − \$300) over unbadged sellers with 2025 average monthly revenue ≥ R in your sample, × 20 for all 40,000 sellers.

In [ ]:
# Your code here


### Decide again

In [ ]:
TEAM = ""                  # e.g. "Team 3: Ana & Ben"  (use the same name every time)
MIN_MONTHLY_REVENUE = None # badge unbadged sellers with 2025 avg monthly revenue >= this ($); 999999 = badge no one  (e.g. 2000)
PREDICTED_PROFIT = None    # in dollars, as a number, e.g. 150000
JUSTIFICATION = """
Three sentences for the decision maker: what you found, what you recommend, and why.
"""

# Stops with a message if anything is still blank, so you don't submit an empty field.
blank = [name for name, value in [("TEAM", TEAM), ("MIN_MONTHLY_REVENUE", MIN_MONTHLY_REVENUE), ("PREDICTED_PROFIT", PREDICTED_PROFIT)] if value in (None, "")]
if blank:
    raise ValueError("Fill these in before submitting: " + ", ".join(blank))
PREDICTED_PROFIT = float(PREDICTED_PROFIT)   # must be a number
print("Enter exactly these values in the submission form:")
print()
print(f"team:              {TEAM}")
print("case:              4")
print("round:             2  (Part II)")
print(f"decision:          {MIN_MONTHLY_REVENUE}")
print(f"predicted_profit:  {PREDICTED_PROFIT:.0f}")

## Submit (round 2)

**[Open the submission form](https://docs.google.com/forms/d/e/1FAIpQLSf-87eDsEHRVat_a4le91sC7bfZFel9fAHNmMtHVuGJzqaHOQ/viewform)** and enter the values printed above, exactly as shown.

- **Predicted profit:** a number such as `150000` (`$150,000` and `150K` also work, but not words). To badge no one, enter `999999` as the decision.
- **Case:** `4`.  **Round:** `2` (Part II).
- **Submissions are final.** You can't change an answer after you submit, so check the printed values first. If your team submits more than once for the same round, only the **first** submission counts.
- Use the **same team name** in every round.

## Reflection (submit)

1. Your Part 1 analysis may have fit the data very well. Why was it still the wrong tool for this decision?
2. In one sentence: what was the difference between **predicting** the outcome and **deciding** what to do?
3. Name one other business decision where the same trap could appear.